# 03 · Preparação dos dados para agrupamento — IBM HR Analytics

**CRISP-DM — preparação dos dados.** Este notebook utiliza a base de RH curada pelo notebook 02 e o dicionário do notebook 00 para construir uma matriz exclusivamente numérica de atributos, apropriada à comparação de algoritmos de clustering no notebook 04.

**Decisões:** as variáveis ordinais mantêm sua ordem e são escalonadas para [0, 1]; as nominais recebem one-hot encoding; as quantitativas recebem MinMaxScaler. `Desligamento` é reservado exclusivamente à análise posterior dos segmentos, não participando do agrupamento. Identificadores e atributos constantes são excluídos.

**Redundância:** por padrão, os atributos derivados (`faixa_etaria`, `categoria_tempo_empresa`, `alta_mobilidade_carreira`) não entram no clustering porque duplicam informações já representadas por `Idade`, `Anos_na_Empresa` e `Num_Empresas_Trabalhadas`. Podem ser utilizados na caracterização posterior dos clusters. A escolha pode ser revista em experimentos documentados.

Saídas: `base_rh_preprocessed.csv` e `base_rh_perfil.csv`, preservando a correspondência entre linhas para análise de segmentos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

BASE = Path('../data/processed')
ARQUIVO_TRATADO = BASE / 'base_rh_tratado.csv'
ARQUIVO_DICIONARIO = BASE / 'dicionario.csv'
ARQUIVO_MATRIZ = BASE / 'base_rh_preprocessed.csv'
ARQUIVO_PERFIL = BASE / 'base_rh_perfil.csv'

In [2]:
for arquivo in (ARQUIVO_TRATADO, ARQUIVO_DICIONARIO):
    if not arquivo.is_file():
        raise FileNotFoundError(f'Arquivo de entrada não encontrado: {arquivo}')

df = pd.read_csv(ARQUIVO_TRATADO, sep=';')
dicionario = pd.read_csv(ARQUIVO_DICIONARIO, sep=';')
if df.empty or df.columns.duplicated().any():
    raise ValueError('Base vazia ou com nomes de colunas duplicados.')
if dicionario['variavel'].duplicated().any():
    raise ValueError('Dicionário contém variáveis duplicadas.')
print(f'Base tratada: {df.shape[0]} registros, {df.shape[1]} colunas')

Base tratada: 1470 registros, 33 colunas


In [3]:
# Exclusões explícitas: não incorporar identificadores, constantes ou desfecho ao clustering.
COLUNAS_EXCLUIDAS = {
    'Numero_do_Funcionario',
    'Contagem_de_Funcionarios', 'Horas_Padrao', 'Maior_de_18',
    'Desligamento',
}
COLUNAS_DERIVADAS = {
    'faixa_etaria', 'categoria_tempo_empresa', 'alta_mobilidade_carreira',
}

# Manter as originais evita contagem implícita em dobro dessas dimensões.
COLUNAS_EXCLUIDAS |= COLUNAS_DERIVADAS

# Não usar atributos constantes, mesmo que tenham novos nomes.
COLUNAS_EXCLUIDAS |= set(dicionario.loc[dicionario['Tipo'] == 'Constante', 'variavel'])
COLUNAS_EXCLUIDAS |= {c for c in df.columns if df[c].nunique(dropna=False) <= 1}

if 'Desligamento' not in df.columns:
    raise ValueError('A coluna Desligamento é necessária para a análise posterior dos clusters.')

# Salva os dados de interpretação separadamente, na MESMA ordem das linhas de entrada.
perfil = df.copy()

In [4]:
tipos = dicionario.set_index('variavel')['Tipo'].to_dict()
COLUNAS_MODELO = [c for c in df.columns if c not in COLUNAS_EXCLUIDAS]
nao_documentadas = [c for c in COLUNAS_MODELO if c not in tipos]
if nao_documentadas:
    raise ValueError(f'Colunas sem classificação no dicionário: {nao_documentadas}')

quantitativas = [c for c in COLUNAS_MODELO if tipos[c] in ('Contínua', 'Discreta')]
ordinais = [c for c in COLUNAS_MODELO if tipos[c] == 'Ordinal']
nominais = [c for c in COLUNAS_MODELO if tipos[c] == 'Nominal']
nao_classificadas = set(COLUNAS_MODELO) - set(quantitativas + ordinais + nominais)
if nao_classificadas:
    raise ValueError(f'Colunas sem transformador apropriado: {sorted(nao_classificadas)}')
if not COLUNAS_MODELO:
    raise ValueError('Nenhuma coluna disponível para clustering.')

print('Atributos: quantitativas =', len(quantitativas),
      '| ordinais =', len(ordinais), '| nominais =', len(nominais))
print('Atributos excluídos:', sorted(c for c in COLUNAS_EXCLUIDAS if c in df.columns))

Atributos: quantitativas = 14 | ordinais = 9 | nominais = 7
Atributos excluídos: ['Desligamento', 'alta_mobilidade_carreira', 'categoria_tempo_empresa']


In [5]:
# As escalas ordinais do notebook 00 já são códigos numéricos ordenados.
# Verificamos explicitamente antes de aplicar MinMaxScaler, sem impor ordem alfabética.
for coluna in quantitativas + ordinais:
    valores = pd.to_numeric(df[coluna], errors='coerce')
    if valores.isna().any() or not np.isfinite(valores.to_numpy()).all():
        raise ValueError(f'Valores não numéricos, ausentes ou infinitos em {coluna}')
    df[coluna] = valores

if df[COLUNAS_MODELO].isna().any().any():
    raise ValueError('Existem valores ausentes nos atributos do clustering.')

# Os níveis efetivamente observados devem ser consistentes com escala.csv, se disponível.
ARQUIVO_ESCALA = BASE / 'escala.csv'
if ordinais and ARQUIVO_ESCALA.is_file():
    escalas = pd.read_csv(ARQUIVO_ESCALA, sep=';')
    for coluna in ordinais:
        definidos = pd.to_numeric(escalas.loc[escalas['coluna'] == coluna, 'nivel'], errors='coerce')
        if definidos.empty or definidos.isna().any():
            raise ValueError(f'Escala ordinal inválida ou ausente: {coluna}')
        invalidos = set(df[coluna].dropna()) - set(definidos)
        if invalidos:
            raise ValueError(f'Valores fora da escala para {coluna}: {invalidos}')

In [6]:
# Quantitativas e ordinais são normalizadas individualmente para [0, 1].
# Nominais são codificadas em colunas binárias; a escolha de ponderações e
# métricas de distância será avaliada na etapa de modelagem (notebook 04).
transformadores = []
if quantitativas:
    transformadores.append(('quantitativas', MinMaxScaler(), quantitativas))
if ordinais:
    transformadores.append(('ordinais', MinMaxScaler(), ordinais))
if nominais:
    transformadores.append(('nominais', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominais))

preprocessor = ColumnTransformer(
    transformers=transformadores,
    remainder='drop',
    verbose_feature_names_out=False,
).set_output(transform='pandas')

In [7]:
X = df[COLUNAS_MODELO].copy()
X_preprocessed = preprocessor.fit_transform(X)

# Invariantes de saída: mesma quantidade e ordem de linhas, dados numéricos,
# sem ausências e sem desfecho/identificador no modelo.
if len(X_preprocessed) != len(df):
    raise AssertionError('A transformação alterou o número de registros.')
if X_preprocessed.columns.duplicated().any():
    raise AssertionError('Nomes de atributos duplicados após codificação.')
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_preprocessed.dtypes):
    raise AssertionError('A matriz contém variáveis não numéricas.')
matriz = X_preprocessed.to_numpy(dtype=float)
if not np.isfinite(matriz).all():
    raise AssertionError('A matriz contém valores ausentes ou infinitos.')
if ((matriz < -1e-12) | (matriz > 1 + 1e-12)).any():
    raise AssertionError('Valores fora do intervalo esperado [0,1].')
if any('Desligamento' in coluna or 'Numero_do_Funcionario' in coluna
       for coluna in X_preprocessed.columns):
    raise AssertionError('Desfecho ou identificador indevidamente presente na matriz.')

BASE.mkdir(parents=True, exist_ok=True)
X_preprocessed.to_csv(ARQUIVO_MATRIZ, sep=';', index=False)
perfil.to_csv(ARQUIVO_PERFIL, sep=';', index=False)

# Releitura confirma integridade dos artefatos exportados.
matriz_salva = pd.read_csv(ARQUIVO_MATRIZ, sep=';')
perfil_salvo = pd.read_csv(ARQUIVO_PERFIL, sep=';')
assert matriz_salva.shape == X_preprocessed.shape
assert len(perfil_salvo) == len(df)
assert matriz_salva.columns.tolist() == X_preprocessed.columns.tolist()
assert matriz_salva.notna().all().all()
print(f'Matriz de clustering: {matriz_salva.shape[0]} registros x {matriz_salva.shape[1]} atributos')
print(f'Matriz: {ARQUIVO_MATRIZ}')
print(f'Base de interpretação: {ARQUIVO_PERFIL}')

Matriz de clustering: 1470 registros x 51 atributos
Matriz: ..\data\processed\base_rh_preprocessed.csv
Base de interpretação: ..\data\processed\base_rh_perfil.csv
